# 03 — Build Performance Analysis
**Purpose:** Critical path analysis, build simulation, bottleneck identification.
**Inputs:** `target_metrics.parquet`, `edge_list.parquet`, `build_schedule.parquet` (optional)
**Outputs:** `data/intermediate/critical_path.parquet`

In [ ]:
from __future__ import annotations

import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split

from buildanalysis.build import (
    compute_critical_path,
    simulate_build,
    validate_simulation,
    whatif_reduce_target_time,
    whatif_remove_edge,
)
from buildanalysis.graphs import build_dependency_graph, compute_centrality_metrics
from buildanalysis.loading import BuildDataset

warnings.filterwarnings("ignore", category=FutureWarning)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams.update({"figure.figsize": (10, 6), "figure.dpi": 100})

DATA_DIR = Path("../../data/processed")
INTERMEDIATE_DIR = DATA_DIR / "intermediate"
INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)

ds = BuildDataset(DATA_DIR, intermediate_dir=INTERMEDIATE_DIR, validate=False)
tm = ds.target_metrics
el = ds.edge_list

# Load target_features from notebook 02 for team/segment info
tf = ds.load_intermediate("target_features")

print(f"target_metrics: {tm.shape[0]:,} targets")
print(f"edge_list:      {el.shape[0]:,} edges")

## 1. Build the Dependency Graph

In [ ]:
bg = build_dependency_graph(tm, el, direct_only=True)
print(f"Graph: {bg.n_targets} targets, {bg.n_edges} direct edges")

## 2. Critical Path Analysis

In [ ]:
cp = compute_critical_path(bg, tm)

print("CRITICAL PATH")
print("=" * 80)
print(f"Critical path length:  {cp.total_time_s:,.1f} s ({cp.total_time_s / 60:.1f} min)")
print(f"Total work (all CPUs): {cp.total_work_s:,.1f} s ({cp.total_work_s / 60:.1f} min)")
print(f"Parallelism ratio:     {cp.parallelism_ratio:.1f}x")
print(f"Targets on critical path: {len(cp.path)}")
print()

# Print the critical path with cumulative time
cumulative = 0.0
print(f"{'#':<4} {'Target':<50} {'Build (ms)':>12} {'Cumulative (ms)':>16}")
print("-" * 86)
for i, target in enumerate(cp.path, 1):
    row = cp.target_slack[cp.target_slack["cmake_target"] == target].iloc[0]
    bt = row["build_time_ms"]
    cumulative += bt
    print(f"{i:<4} {target:<50} {bt:>12,.0f} {cumulative:>16,.0f}")

In [ ]:
# Bar chart of critical path targets
cp_targets = cp.target_slack[cp.target_slack["on_critical_path"]].sort_values("earliest_start_ms")

fig, ax = plt.subplots(figsize=(12, max(6, len(cp_targets) * 0.3)))
ax.barh(cp_targets["cmake_target"], cp_targets["build_time_ms"] / 1000, color="steelblue", edgecolor="white", linewidth=0.3)
ax.set_xlabel("Build time (s)")
ax.set_ylabel("Target")
ax.set_title("Critical Path Targets — Build Time")
ax.invert_yaxis()
fig.tight_layout()
plt.show()

In [ ]:
# Slack distribution
slack = cp.target_slack["slack_ms"]
cp_length_ms = cp.total_time_s * 1000
near_zero_threshold = cp_length_ms * 0.05

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(slack, bins=60, edgecolor="white", linewidth=0.3, color="coral")
ax.axvline(near_zero_threshold, color="red", linestyle="--", alpha=0.7, label=f"5% of CP ({near_zero_threshold:,.0f} ms)")
ax.set_xlabel("Slack (ms)")
ax.set_ylabel("Count")
ax.set_title("Slack Distribution Across All Targets")
ax.legend()
fig.tight_layout()
plt.show()

n_zero = (slack.abs() < 1e-9).sum()
n_near_zero = (slack < near_zero_threshold).sum()
print(f"Targets with zero slack (on critical path):   {n_zero}")
print(f"Targets with near-zero slack (<5% of CP):     {n_near_zero}")
print(f"Targets with significant slack:               {len(slack) - n_near_zero}")

## 3. Build Scheduling Simulation

In [ ]:
core_counts = [8, 16, 32, 64, 128]
sim_results = []

for n_cores in core_counts:
    sched = simulate_build(bg, tm, n_cores=n_cores)
    wall_ms = sched["end_ms"].max()
    total_work_ms = (sched["end_ms"] - sched["start_ms"]).sum()
    utilisation = total_work_ms / (wall_ms * n_cores) if wall_ms > 0 else 0.0
    sim_results.append({
        "cores": n_cores,
        "wall_time_ms": wall_ms,
        "wall_time_s": wall_ms / 1000,
        "utilisation": utilisation,
    })
    print(f"  {n_cores:>3} cores: {wall_ms / 1000:>8.1f} s  (util: {utilisation:.1%})")

sim_df = pd.DataFrame(sim_results)

# Diminishing returns: where does doubling cores yield <10% improvement?
for i in range(1, len(sim_df)):
    prev = sim_df.iloc[i - 1]["wall_time_s"]
    curr = sim_df.iloc[i]["wall_time_s"]
    improvement = (prev - curr) / prev * 100
    if improvement < 10:
        print(f"\nDiminishing returns after {sim_df.iloc[i - 1]['cores']:.0f} cores "
              f"({sim_df.iloc[i]['cores']:.0f} cores only {improvement:.1f}% faster)")
        break

In [ ]:
# Wall time vs core count with critical path floor
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(sim_df["cores"], sim_df["wall_time_s"], marker="o", linewidth=2, label="Simulated wall time")
ax.axhline(cp.total_time_s, color="red", linestyle="--", alpha=0.7, label=f"Critical path floor ({cp.total_time_s:.1f} s)")
ax.set_xlabel("Core count")
ax.set_ylabel("Wall time (s)")
ax.set_title("Simulated Build Wall Time vs Core Count")
ax.set_xscale("log", base=2)
ax.set_xticks(core_counts)
ax.set_xticklabels([str(c) for c in core_counts])
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
# Validate against observed build schedule if available
if ds.has_file("build_schedule"):
    observed = ds.build_schedule
    # Aggregate observed schedule to target level for comparison
    obs_target = observed.groupby("cmake_target").agg(
        start_ms=("start_time_ms", "min"),
        end_ms=("end_time_ms", "max"),
    ).reset_index()

    # Simulate at observed core count (infer from overlapping tasks)
    # Use a reasonable default if we can't infer
    sim_for_validation = simulate_build(bg, tm, n_cores=16)
    validation = validate_simulation(sim_for_validation, obs_target)

    print("SIMULATION VALIDATION (16 cores)")
    print("=" * 50)
    print(f"  Simulated wall time: {validation['simulated_wall_time_ms'] / 1000:.1f} s")
    print(f"  Observed wall time:  {validation['observed_wall_time_ms'] / 1000:.1f} s")
    print(f"  Error:               {validation['wall_time_error_pct']:.1f}%")
    print(f"  Within tolerance:    {validation['within_tolerance']}")
else:
    print("build_schedule.parquet not found — skipping validation against observed data.")

## 4. Bottleneck Identification

In [ ]:
# Top 20 targets by build time with critical path and slack info
slack_map = cp.target_slack.set_index("cmake_target")[["on_critical_path", "slack_ms"]]

# Merge segment name from target_features if available
segment_col = tf[["cmake_target", "segment_name"]].copy() if "segment_name" in tf.columns else None

top20_bt = (
    tm[["cmake_target", "target_type", "total_build_time_ms"]]
    .dropna(subset=["total_build_time_ms"])
    .sort_values("total_build_time_ms", ascending=False)
    .head(20)
    .copy()
)
top20_bt = top20_bt.merge(slack_map, left_on="cmake_target", right_index=True, how="left")
if segment_col is not None:
    top20_bt = top20_bt.merge(segment_col, on="cmake_target", how="left")

top20_bt["build_time_s"] = (top20_bt["total_build_time_ms"] / 1000).round(1)
top20_bt.index = range(1, len(top20_bt) + 1)

print("TOP 20 TARGETS BY BUILD TIME")
print("=" * 110)
display_cols = ["cmake_target", "target_type", "build_time_s", "on_critical_path", "slack_ms"]
if segment_col is not None:
    display_cols.append("segment_name")
print(top20_bt[display_cols].to_string())

In [ ]:
# Top 20 targets by critical path contribution (build time of CP targets)
cp_contribution = cp.target_slack[cp.target_slack["on_critical_path"]].copy()
cp_contribution = cp_contribution.sort_values("build_time_ms", ascending=False).head(20)
cp_contribution["pct_of_cp"] = (100 * cp_contribution["build_time_ms"] / (cp.total_time_s * 1000)).round(1)
cp_contribution.index = range(1, len(cp_contribution) + 1)

print("TOP 20 TARGETS BY CRITICAL PATH CONTRIBUTION")
print("=" * 80)
print(cp_contribution[["cmake_target", "build_time_ms", "pct_of_cp"]].to_string())

In [ ]:
# Phase-dominated targets
phase_checks = [
    ("compile_time_sum_ms", 0.8, "COMPILE-DOMINATED"),
    ("link_time_ms", 0.3, "LINK-DOMINATED"),
    ("codegen_time_ms", 0.3, "CODEGEN-DOMINATED"),
]

for col, threshold, label in phase_checks:
    if col not in tm.columns:
        print(f"{label}: column '{col}' not available — skipping")
        continue
    bt = tm["total_build_time_ms"]
    mask = (tm[col] > bt * threshold) & bt.notna() & tm[col].notna() & (bt > 0)
    dominated = tm.loc[mask, ["cmake_target", "target_type", col, "total_build_time_ms"]].copy()
    dominated["ratio"] = (dominated[col] / dominated["total_build_time_ms"]).round(2)
    dominated = dominated.sort_values(col, ascending=False)
    print(f"\n{label} targets ({col} > {threshold:.0%} of total_build_time_ms): {len(dominated)}")
    if len(dominated) > 0:
        print(dominated.head(10).to_string(index=False))

## 5. Compilation Cost Regression

In [ ]:
# Predict compile_time_sum_ms from structural features
feature_cols = [
    "code_lines_total",
    "preprocessed_bytes_total",
    "file_count",
    "gcc_template_time_sum_ms",
    "codegen_ratio",
    "total_dependency_count",
    "header_depth_max",
]
target_col = "compile_time_sum_ms"

available_features = [c for c in feature_cols if c in tm.columns]
reg_df = tm[["cmake_target"] + available_features + [target_col]].dropna().copy()
print(f"Regression dataset: {len(reg_df)} targets, {len(available_features)} features")

X = reg_df[available_features]
y = reg_df[target_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = GradientBoostingRegressor(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    random_state=42,
)
model.fit(X_train, y_train)

r2_train = model.score(X_train, y_train)
r2_test = model.score(X_test, y_test)
print(f"R² (train): {r2_train:.3f}")
print(f"R² (test):  {r2_test:.3f}")

In [ ]:
# Feature importance (permutation-based on test set)
perm_imp = permutation_importance(model, X_test, y_test, n_repeats=10, random_state=42)
imp_df = pd.DataFrame({
    "feature": available_features,
    "importance_mean": perm_imp.importances_mean,
    "importance_std": perm_imp.importances_std,
}).sort_values("importance_mean", ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(imp_df["feature"], imp_df["importance_mean"], xerr=imp_df["importance_std"],
        color="steelblue", edgecolor="white", linewidth=0.3)
ax.set_xlabel("Permutation importance (drop in R²)")
ax.set_title("Feature Importance for Compile Time Prediction")
ax.invert_yaxis()
fig.tight_layout()
plt.show()

print("FEATURE IMPORTANCE")
print("=" * 60)
print(imp_df.to_string(index=False))

In [ ]:
# Predicted vs actual compile time
y_pred_all = model.predict(reg_df[available_features])
reg_df["predicted_ms"] = y_pred_all
reg_df["residual_ms"] = reg_df[target_col] - reg_df["predicted_ms"]

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(reg_df[target_col], reg_df["predicted_ms"], alpha=0.4, s=15)
max_val = max(reg_df[target_col].max(), reg_df["predicted_ms"].max())
ax.plot([0, max_val], [0, max_val], color="red", linestyle="--", alpha=0.7, label="Perfect prediction")
ax.set_xlabel("Actual compile time (ms)")
ax.set_ylabel("Predicted compile time (ms)")
ax.set_title(f"Predicted vs Actual Compile Time (R² test = {r2_test:.3f})")
ax.legend()
fig.tight_layout()
plt.show()

# Anomaly detection: residuals > 2 std deviations (much slower than expected)
residual_std = reg_df["residual_ms"].std()
anomalies = reg_df[reg_df["residual_ms"] > 2 * residual_std].sort_values("residual_ms", ascending=False)
print(f"\nANOMALIES: targets much slower than predicted (residual > 2σ = {2 * residual_std:,.0f} ms)")
print("=" * 90)
if len(anomalies) > 0:
    print(anomalies[["cmake_target", target_col, "predicted_ms", "residual_ms"]].head(20).to_string(index=False))
else:
    print("No anomalies detected.")

## 6. What-If Analysis

In [ ]:
# What-if: remove high-betweenness edges
centrality = compute_centrality_metrics(bg)

# Get edges with betweenness centrality (use edge betweenness)
edge_betweenness = nx.edge_betweenness_centrality(bg.graph)
top_edges = sorted(edge_betweenness.items(), key=lambda x: x[1], reverse=True)[:20]

edge_removal_results = []
for (src, dep), eb in top_edges:
    result = whatif_remove_edge(bg, tm, src, dep)
    result["edge_betweenness"] = eb
    edge_removal_results.append(result)

edge_df = pd.DataFrame(edge_removal_results)
edge_df["source"] = edge_df["removed_edge"].apply(lambda x: x[0])
edge_df["dependency"] = edge_df["removed_edge"].apply(lambda x: x[1])
edge_df = edge_df.sort_values("delta_ms", ascending=False)
edge_df.index = range(1, len(edge_df) + 1)

print("WHAT-IF: REMOVE HIGH-BETWEENNESS EDGES")
print("=" * 100)
print(edge_df[["source", "dependency", "edge_betweenness", "baseline_critical_path_ms",
               "new_critical_path_ms", "delta_ms", "is_valid"]].to_string())

In [ ]:
# What-if: reduce compile time by 50% for top 10 critical-path targets
cp_top10 = (
    cp.target_slack[cp.target_slack["on_critical_path"]]
    .sort_values("build_time_ms", ascending=False)
    .head(10)
)

reduction_results = []
for _, row in cp_top10.iterrows():
    target = row["cmake_target"]
    result = whatif_reduce_target_time(bg, tm, target, reduction_pct=50.0)
    reduction_results.append(result)

red_df = pd.DataFrame(reduction_results).sort_values("delta_ms", ascending=False)
red_df.index = range(1, len(red_df) + 1)

print("WHAT-IF: 50% COMPILE TIME REDUCTION ON TOP CP TARGETS")
print("=" * 100)
print(red_df[["target", "original_time_ms", "reduced_time_ms",
              "baseline_critical_path_ms", "new_critical_path_ms", "delta_ms",
              "on_original_critical_path"]].to_string())

In [ ]:
# Combined what-if bar chart: top 10 improvements across both analyses
combined = []

for _, row in edge_df.head(10).iterrows():
    if row["delta_ms"] > 0:
        combined.append({
            "intervention": f"Remove edge: {row['source']} -> {row['dependency']}",
            "delta_ms": row["delta_ms"],
            "type": "Edge removal",
        })

for _, row in red_df.head(10).iterrows():
    if row["delta_ms"] > 0:
        combined.append({
            "intervention": f"50% faster: {row['target']}",
            "delta_ms": row["delta_ms"],
            "type": "Time reduction",
        })

combined_df = pd.DataFrame(combined).sort_values("delta_ms", ascending=False).head(10)

if len(combined_df) > 0:
    fig, ax = plt.subplots(figsize=(12, max(6, len(combined_df) * 0.5)))
    colors = ["coral" if t == "Edge removal" else "steelblue" for t in combined_df["type"]]
    ax.barh(combined_df["intervention"], combined_df["delta_ms"] / 1000, color=colors, edgecolor="white", linewidth=0.3)
    ax.set_xlabel("Critical path reduction (s)")
    ax.set_title("Top 10 What-If Improvements")
    ax.invert_yaxis()

    # Legend
    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(color="coral", label="Edge removal"), Patch(color="steelblue", label="Time reduction")])

    fig.tight_layout()
    plt.show()
else:
    print("No what-if improvements found (all deltas <= 0).")

## 7. Persist Output

In [ ]:
# Save critical_path.parquet with per-target slack data
critical_path_df = cp.target_slack.copy()
critical_path_df["critical_path_contribution_ms"] = critical_path_df.apply(
    lambda r: r["build_time_ms"] if r["on_critical_path"] else 0.0, axis=1
)

out_path = ds.save_intermediate("critical_path", critical_path_df)
print(f"Saved critical_path.parquet: {critical_path_df.shape[0]} rows x {critical_path_df.shape[1]} cols")
print(f"  -> {out_path}")

# Update critical_path_contribution_ms in target_metrics if column exists
if "critical_path_contribution_ms" in tm.columns:
    cp_map = critical_path_df.set_index("cmake_target")["critical_path_contribution_ms"]
    tm["critical_path_contribution_ms"] = tm["cmake_target"].map(cp_map).fillna(0).astype(int)
    tm.to_parquet(DATA_DIR / "target_metrics.parquet", index=False)
    print("Updated critical_path_contribution_ms in target_metrics.parquet")